In [1]:
import pandas as pd

from src.inference import (
    load_model_artifacts,
    predict_failure,
    predict_single_machine
)

In [2]:
#Loading model and its configuration
model, config = load_model_artifacts()
config

{'model_name': 'Tuned Random Forest', 'threshold': np.float64(0.3)}

#### Predicting a single machine

In [3]:
single_prediction = predict_single_machine(
    machine_type="M",
    air_temperature=298.1,
    process_temperature=308.6,
    rotational_speed=1551,
    torque=42.8,
    tool_wear=120,
    model=model,
    threshold=config["threshold"]
)

single_prediction

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],failure_probability,predicted_failure
0,M,298.1,308.6,1551,42.8,120,0.0,0


The model receives raw machine sensor readings and returns a failure probability.
The final class prediction is based on the selected threshold from validation-set analysis.

#### Batch prediction example

In [4]:
df = pd.read_csv("../data/raw/ai4i2020.csv")

In [5]:
X = df.drop(columns=[
    "UDI",
    "Product ID",
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
])

y = df["Machine failure"]

In [6]:
sample_input = X.sample(10, random_state=42)
sample_predictions = predict_failure(
    sample_input,
    model=model,
    threshold=config["threshold"]
)

sample_predictions

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],failure_probability,predicted_failure
6252,L,300.8,310.3,1538,36.1,198,0.002,0
4684,M,303.6,311.8,1421,44.8,101,0.600,1
1731,M,298.3,307.9,1485,42.0,117,0.000,0
4742,L,303.3,311.3,1592,33.7,14,0.000,0
4521,L,302.4,310.4,1865,23.9,129,0.000,0
6340,H,300.5,309.9,1397,45.9,210,0.778,1
576,H,297.7,309.7,1440,51.1,191,0.004,0
5202,L,303.7,312.7,1335,51.1,161,0.002,0
6363,M,300.0,309.6,1618,36.2,53,0.000,0
439,M,297.4,308.3,1535,34.6,51,0.000,0


In [7]:
sample_predictions.to_csv(
    "../reports/sample_predictions.csv",
    index=False
)